In [2]:
import pandas as pd  # Needed for reading dataset

data = pd.read_csv("SMSSpamCollection", sep='\t', names=["label", "message"])
data['label_num'] = data['label'].map({'ham': 0, 'spam': 1})

print("Dataset loaded successfully!")

print(data.head())


Dataset loaded successfully!
  label                                            message  label_num
0   ham  Go until jurong point, crazy.. Available only ...          0
1   ham                      Ok lar... Joking wif u oni...          0
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...          1
3   ham  U dun say so early hor... U c already then say...          0
4   ham  Nah I don't think he goes to usf, he lives aro...          0


In [3]:
data.shape

(5572, 3)

In [5]:
# Step 2: Text Preprocessing
import re  # For text cleaning (regex)
import nltk  # For tokenization, stopwords, stemming
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

# Download required NLTK resources
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

# Use a list instead of set
stop_words = stopwords.words('english')  # list of English stopwords
stemmer = PorterStemmer()  # Stemmer to reduce words to root form

# Simple text cleaning function
def clean_text_simple(text):
    text = text.lower()  # lowercase
    text = re.sub(r"http\S+|www\S+", "", text)  # remove URLs
    text = re.sub(r"[^a-z\s]", "", text)  # remove punctuation/numbers
    words = word_tokenize(text)  # tokenize
    words = [stemmer.stem(w) for w in words if w not in stop_words]  # remove stopwords + stem
    return " ".join(words)

# Apply cleaning to all messages
data["clean_msg"] = data["message"].apply(clean_text_simple)

# Show first 5 messages
print(data[["message", "clean_msg"]].head())

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\adity\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\adity\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\adity\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


                                             message  \
0  Go until jurong point, crazy.. Available only ...   
1                      Ok lar... Joking wif u oni...   
2  Free entry in 2 a wkly comp to win FA Cup fina...   
3  U dun say so early hor... U c already then say...   
4  Nah I don't think he goes to usf, he lives aro...   

                                           clean_msg  
0  go jurong point crazi avail bugi n great world...  
1                              ok lar joke wif u oni  
2  free entri wkli comp win fa cup final tkt st m...  
3                u dun say earli hor u c alreadi say  
4          nah dont think goe usf live around though  


In [6]:
# Step 3: Train-Test Split
from sklearn.model_selection import train_test_split

X = data["clean_msg"]
y = data["label_num"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [7]:
# Step 4: TF-IDF Feature Extraction
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=3000)  # keep top 3000 words
X_train_tfidf = tfidf.fit_transform(X_train)  # learn vocab & transform training data
X_test_tfidf = tfidf.transform(X_test)  # transform test data using same vocab

In [8]:
# Step 5: Train Naive Bayes
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train_tfidf, y_train)
y_pred = nb.predict(X_test_tfidf)

In [9]:
# Step 6: Evaluate Model
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred, target_names=["HAM","SPAM"]))

Accuracy: 0.9721973094170404
Precision: 1.0
Recall: 0.7919463087248322
F1 Score: 0.8838951310861424
Confusion Matrix:
 [[966   0]
 [ 31 118]]
Classification Report:
               precision    recall  f1-score   support

         HAM       0.97      1.00      0.98       966
        SPAM       1.00      0.79      0.88       149

    accuracy                           0.97      1115
   macro avg       0.98      0.90      0.93      1115
weighted avg       0.97      0.97      0.97      1115

